# EcoShield AI — Tekil Ağır Model Optimizasyonu

Bu notebook Görev 6 kapsamında ağır modelleri **hafif model filtresi olmadan** optimize eder.

- XGBoost ve CatBoost ortak train splitinin tamamıyla eğitilir.
- Model ve eşik seçimi yalnızca validation splitinde yapılır.
- Test feature, target ve ID dizileri yüklenmez.
- Resampling uygulanmaz; sınıf dengesizliği yalnızca model ağırlıklarıyla ele alınır.
- Her denemede train–validation farkı, süre, RAM, model boyutu ve inference süresi kaydedilir.
- Accuracy ana seçim metriği değildir.

Görev 5 cascade sonuçları bu notebookta model seçmek için kullanılmaz. Görev 7'de seçilen tekil model ile cascade, test splitinde yalnızca bir kez karşılaştırılacaktır.


## 1. Ayarlar ve paket kontrolü

In [ ]:
QUICK_MODE = False
RANDOM_STATE = 42
PREFERRED_DEVICE = "GPU"
MINIMUM_RECALL = 0.90
INFERENCE_SAMPLE_SIZE = 10_000
INFERENCE_REPEATS = 5
CURVE_POINTS_PER_TRIAL = 400

import importlib.util
import json
import os
import sys
import time
import warnings
from pathlib import Path

required = {
    "catboost": "catboost",
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "psutil": "psutil",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
    "xgboost": "xgboost",
}
missing = [pip for module, pip in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError("Eksik paketler: " + ", ".join(missing))

print("QUICK_MODE:", QUICK_MODE)
print("Minimum validation recall:", MINIMUM_RECALL)
print("Tercih edilen cihaz:", PREFERRED_DEVICE)
print("Test split bu notebookta yüklenmeyecek.")


## 2. Importlar, proje yolları ve ortak modül

In [ ]:
import gc
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from xgboost import XGBClassifier

warnings.filterwarnings("default")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    raise FileNotFoundError("notebooks/common_preprocessing.py bulunamadı.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import (
    build_project_paths,
    find_project_root,
    get_or_create_profile_cache,
)

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)
MODEL_DIR = PROJECT_ROOT / "models" / "heavy"
PREDICTIONS_DIR = PATHS.outputs / "predictions"
FIGURES_DIR = PATHS.outputs / "figures"
for directory in [MODEL_DIR, PREDICTIONS_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Proje kökü:", PROJECT_ROOT)
print("XGBoost:", xgb.__version__)
print("Başlangıç RAM:", f"{psutil.Process(os.getpid()).memory_info().rss / (1024**3):.2f} GB")


## 3. Değerlendirme ve threshold yardımcıları

In [ ]:
def ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)


def select_rows(X, row_count):
    row_count = min(row_count, X.shape[0])
    if hasattr(X, "iloc"):
        return X.iloc[:row_count]
    return X[:row_count]


def build_threshold_table(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=np.int8)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if not np.isfinite(probabilities).all():
        raise ValueError("Olasılıklar NaN veya sonsuz değer içeriyor.")

    order = np.argsort(-probabilities, kind="mergesort")
    sorted_probabilities = probabilities[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target, dtype=np.int64)
    group_end_mask = np.r_[sorted_probabilities[:-1] != sorted_probabilities[1:], True]
    group_end_indices = np.flatnonzero(group_end_mask)

    thresholds = sorted_probabilities[group_end_indices]
    predicted_positive = group_end_indices + 1
    tp = cumulative_tp[group_end_indices]
    fp = predicted_positive - tp
    positive_count = int(y_true.sum())
    negative_count = len(y_true) - positive_count
    fn = positive_count - tp
    tn = negative_count - fp
    precision = np.divide(
        tp, tp + fp,
        out=np.zeros_like(tp, dtype=np.float64),
        where=(tp + fp) > 0,
    )
    recall = tp / positive_count
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision),
        where=(precision + recall) > 0,
    )
    return pd.DataFrame({
        "threshold": thresholds,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "predicted_fraud_count": predicted_positive,
        "positive_prediction_rate": predicted_positive / len(y_true),
    })


def select_operating_point(table):
    eligible = table[table["recall"] >= MINIMUM_RECALL]
    if eligible.empty:
        selected = table.sort_values(
            ["recall", "f1", "precision"], ascending=[False, False, False]
        ).iloc[0]
        return selected, False, "Minimum recall sağlanamadı; en yüksek recall"
    selected = eligible.sort_values(
        ["f1", "precision", "recall", "threshold"],
        ascending=[False, False, False, False],
    ).iloc[0]
    return (
        selected,
        True,
        f"Validation recall >= {MINIMUM_RECALL:.2f} şartında maksimum F1",
    )


def measure_inference(model, X_validation):
    sample = select_rows(X_validation, INFERENCE_SAMPLE_SIZE)
    _ = model.predict_proba(sample)[:, 1]
    durations = []
    for _ in tqdm(range(INFERENCE_REPEATS), desc="10K inference", leave=False):
        started = time.perf_counter()
        _ = model.predict_proba(sample)[:, 1]
        durations.append(time.perf_counter() - started)
    return float(np.mean(durations)), float(np.std(durations)), int(sample.shape[0])


def downsample_curve(table):
    if len(table) <= CURVE_POINTS_PER_TRIAL:
        return table.copy()
    indices = np.unique(
        np.linspace(0, len(table) - 1, CURVE_POINTS_PER_TRIAL).astype(int)
    )
    return table.iloc[indices].copy()


def evaluate_trial(
    *, model_family, trial_name, model, data, params, device,
    training_seconds, ram_before, ram_after,
):
    started = time.perf_counter()
    validation_probabilities = model.predict_proba(data["X_validation"])[:, 1]
    validation_inference_seconds = time.perf_counter() - started

    started = time.perf_counter()
    train_probabilities = model.predict_proba(data["X_train"])[:, 1]
    train_inference_seconds = time.perf_counter() - started

    validation_pr_auc = float(
        average_precision_score(data["y_validation"], validation_probabilities)
    )
    validation_roc_auc = float(
        roc_auc_score(data["y_validation"], validation_probabilities)
    )
    train_pr_auc = float(
        average_precision_score(data["y_train"], train_probabilities)
    )
    train_roc_auc = float(
        roc_auc_score(data["y_train"], train_probabilities)
    )

    threshold_table = build_threshold_table(
        data["y_validation"], validation_probabilities
    )
    selected, constraint_met, selection_reason = select_operating_point(threshold_table)
    inference_mean, inference_std, inference_rows = measure_inference(
        model, data["X_validation"]
    )

    metrics = selected.to_dict()
    metrics.update({
        "trial_name": trial_name,
        "model_family": model_family,
        "device": device,
        "split_version": "common_v2",
        "preprocessing_profile": (
            "catboost" if model_family == "CatBoost" else "sklearn_tree"
        ),
        "imbalance_method": params.get("_imbalance_method", "unknown"),
        "recall_constraint_met": bool(constraint_met),
        "threshold_selection_reason": selection_reason,
        "train_pr_auc": train_pr_auc,
        "validation_pr_auc": validation_pr_auc,
        "pr_auc_gap": train_pr_auc - validation_pr_auc,
        "train_roc_auc": train_roc_auc,
        "validation_roc_auc": validation_roc_auc,
        "roc_auc_gap": train_roc_auc - validation_roc_auc,
        "training_seconds": float(training_seconds),
        "train_inference_seconds": float(train_inference_seconds),
        "validation_inference_seconds": float(validation_inference_seconds),
        "inference_seconds_10k_mean": inference_mean,
        "inference_seconds_10k_std": inference_std,
        "inference_sample_rows": inference_rows,
        "ram_before_gb": float(ram_before),
        "ram_after_fit_gb": float(ram_after),
        "best_iteration": int(getattr(model, "best_iteration", -1) or -1),
        "parameters_json": json.dumps(
            {k: v for k, v in params.items() if not k.startswith("_")},
            ensure_ascii=False,
            sort_keys=True,
        ),
    })
    sampled_curve = downsample_curve(threshold_table)
    sampled_curve["trial_name"] = trial_name
    sampled_curve["model_family"] = model_family
    return metrics, validation_probabilities, sampled_curve


## 4. XGBoost optimizasyonu

`sklearn_tree` cache'i kullanılır. Tam dengesizlik ağırlığına ek olarak karekök ağırlık ve daha sığ/regularized yapı denenir. Validation, early stopping ve model seçimi içindir; test yüklenmez.


In [ ]:
tree_data = get_or_create_profile_cache(
    "xgboost", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
assert "X_test" not in tree_data
validation_ids = tree_data["id_validation"].copy()
validation_target = tree_data["y_validation"].copy()

negative_count = int((tree_data["y_train"] == 0).sum())
positive_count = int((tree_data["y_train"] == 1).sum())
full_weight = negative_count / positive_count
sqrt_weight = float(np.sqrt(full_weight))

xgb_trials = [
    {
        "trial_name": "xgb_d8_full_weight",
        "max_depth": 8, "learning_rate": 0.03,
        "min_child_weight": 5, "subsample": 0.8,
        "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "scale_pos_weight": full_weight,
        "_imbalance_method": "full_scale_pos_weight",
    },
    {
        "trial_name": "xgb_d8_sqrt_weight",
        "max_depth": 8, "learning_rate": 0.03,
        "min_child_weight": 8, "subsample": 0.85,
        "colsample_bytree": 0.85, "reg_alpha": 0.2, "reg_lambda": 3.0,
        "scale_pos_weight": sqrt_weight,
        "_imbalance_method": "sqrt_scale_pos_weight",
    },
    {
        "trial_name": "xgb_d6_sqrt_regularized",
        "max_depth": 6, "learning_rate": 0.04,
        "min_child_weight": 10, "subsample": 0.85,
        "colsample_bytree": 0.9, "reg_alpha": 0.5, "reg_lambda": 5.0,
        "scale_pos_weight": sqrt_weight,
        "_imbalance_method": "sqrt_scale_pos_weight",
    },
]
if QUICK_MODE:
    xgb_trials = xgb_trials[:1]

trial_rows = []
curve_frames = []
validation_prediction_columns = {}
trained_candidates = {}

for trial in tqdm(xgb_trials, desc="XGBoost denemeleri", unit=" model"):
    trial_name = trial["trial_name"]
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "n_estimators": 2_500 if not QUICK_MODE else 300,
        "early_stopping_rounds": 100 if not QUICK_MODE else 30,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        **{k: v for k, v in trial.items() if k != "trial_name" and not k.startswith("_")},
    }
    device_attempts = ["cuda", "cpu"] if PREFERRED_DEVICE.upper() == "GPU" else ["cpu"]
    last_error = None
    for device in device_attempts:
        model = XGBClassifier(**params, device=device)
        print(f"\n{trial_name} — cihaz: {device.upper()}")
        ram_before = ram_gb()
        started = time.perf_counter()
        try:
            model.fit(
                tree_data["X_train"], tree_data["y_train"],
                eval_set=[(tree_data["X_validation"], tree_data["y_validation"])],
                verbose=100,
            )
            training_seconds = time.perf_counter() - started
            break
        except Exception as error:
            last_error = error
            print(f"{device.upper()} başarısız: {str(error)[:800]}")
            del model
            gc.collect()
    else:
        raise RuntimeError(f"{trial_name} eğitilemedi.") from last_error

    evaluation_params = {**params, "_imbalance_method": trial["_imbalance_method"]}
    metrics, probabilities, curve = evaluate_trial(
        model_family="XGBoost",
        trial_name=trial_name,
        model=model,
        data=tree_data,
        params=evaluation_params,
        device=device.upper(),
        training_seconds=training_seconds,
        ram_before=ram_before,
        ram_after=ram_gb(),
    )
    trial_rows.append(metrics)
    curve_frames.append(curve)
    validation_prediction_columns[trial_name] = probabilities
    trained_candidates[trial_name] = model
    print(pd.Series(metrics)[[
        "validation_pr_auc", "validation_roc_auc", "f1",
        "precision", "recall", "pr_auc_gap", "training_seconds",
    ]])

print("XGBoost denemeleri tamamlandı. RAM:", f"{ram_gb():.2f} GB")


## 5. XGBoost cache'ini serbest bırakma

In [ ]:
del tree_data
gc.collect()
print("Tree cache bellekten çıkarıldı. RAM:", f"{ram_gb():.2f} GB")


## 6. CatBoost optimizasyonu

Ham kategorik özellikleri koruyan `catboost` profili kullanılır. Dengeli iki farklı yapı ile ağırlıksız bir yapı karşılaştırılır. Ağırlıksız deneme, threshold tuning yapılacağı için ayrıca değerlendirilir.


In [ ]:
catboost_data = get_or_create_profile_cache(
    "catboost", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
assert "X_test" not in catboost_data
assert np.array_equal(validation_ids, catboost_data["id_validation"])
assert np.array_equal(validation_target, catboost_data["y_validation"])

catboost_categorical_columns = catboost_data["X_train"].select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

catboost_trials = [
    {
        "trial_name": "cat_d8_balanced",
        "depth": 8, "learning_rate": 0.03,
        "l2_leaf_reg": 5.0, "random_strength": 1.0,
        "auto_class_weights": "Balanced",
        "_imbalance_method": "auto_class_weights_balanced",
    },
    {
        "trial_name": "cat_d7_balanced_regularized",
        "depth": 7, "learning_rate": 0.04,
        "l2_leaf_reg": 8.0, "random_strength": 0.5,
        "auto_class_weights": "Balanced",
        "_imbalance_method": "auto_class_weights_balanced",
    },
    {
        "trial_name": "cat_d8_unweighted",
        "depth": 8, "learning_rate": 0.03,
        "l2_leaf_reg": 8.0, "random_strength": 0.5,
        "_imbalance_method": "no_class_weight",
    },
]
if QUICK_MODE:
    catboost_trials = catboost_trials[:1]

for trial in tqdm(catboost_trials, desc="CatBoost denemeleri", unit=" model"):
    trial_name = trial["trial_name"]
    params = {
        "iterations": 2_500 if not QUICK_MODE else 300,
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": RANDOM_STATE,
        "use_best_model": True,
        "allow_writing_files": False,
        "verbose": 100,
        **{k: v for k, v in trial.items() if k != "trial_name" and not k.startswith("_")},
    }
    device_attempts = ["GPU", "CPU"] if PREFERRED_DEVICE.upper() == "GPU" else ["CPU"]
    last_error = None
    for device in device_attempts:
        device_params = params.copy()
        device_params["task_type"] = device
        if device == "GPU":
            device_params["devices"] = "0"
        else:
            device_params["thread_count"] = -1
        model = CatBoostClassifier(**device_params)
        print(f"\n{trial_name} — cihaz: {device}")
        ram_before = ram_gb()
        started = time.perf_counter()
        try:
            model.fit(
                catboost_data["X_train"], catboost_data["y_train"],
                cat_features=catboost_categorical_columns,
                eval_set=(
                    catboost_data["X_validation"],
                    catboost_data["y_validation"],
                ),
                early_stopping_rounds=100 if not QUICK_MODE else 30,
            )
            training_seconds = time.perf_counter() - started
            break
        except Exception as error:
            last_error = error
            print(f"{device} başarısız: {str(error)[:800]}")
            del model
            gc.collect()
    else:
        raise RuntimeError(f"{trial_name} eğitilemedi.") from last_error

    evaluation_params = {**params, "_imbalance_method": trial["_imbalance_method"]}
    metrics, probabilities, curve = evaluate_trial(
        model_family="CatBoost",
        trial_name=trial_name,
        model=model,
        data=catboost_data,
        params=evaluation_params,
        device=device,
        training_seconds=training_seconds,
        ram_before=ram_before,
        ram_after=ram_gb(),
    )
    trial_rows.append(metrics)
    curve_frames.append(curve)
    validation_prediction_columns[trial_name] = probabilities
    trained_candidates[trial_name] = model
    print(pd.Series(metrics)[[
        "validation_pr_auc", "validation_roc_auc", "f1",
        "precision", "recall", "pr_auc_gap", "training_seconds",
    ]])

print("CatBoost denemeleri tamamlandı. RAM:", f"{ram_gb():.2f} GB")


## 7. Validation karşılaştırması ve tekil model seçimi

In [ ]:
optimization_results = pd.DataFrame(trial_rows).sort_values(
    [
        "recall_constraint_met",
        "f1",
        "validation_pr_auc",
        "precision",
        "pr_auc_gap",
    ],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)
threshold_curves = pd.concat(curve_frames, ignore_index=True)

eligible = optimization_results[
    optimization_results["recall_constraint_met"]
].copy()
if eligible.empty:
    selected_trial = optimization_results.sort_values(
        ["recall", "f1", "validation_pr_auc"],
        ascending=[False, False, False],
    ).iloc[0]
    model_selection_reason = "Minimum recall sağlanamadı; en yüksek recall"
else:
    selected_trial = eligible.sort_values(
        ["f1", "validation_pr_auc", "precision", "pr_auc_gap"],
        ascending=[False, False, False, True],
    ).iloc[0]
    model_selection_reason = (
        f"Validation recall >= {MINIMUM_RECALL:.2f}; maksimum F1, "
        "ardından PR-AUC, precision ve daha düşük train-validation farkı"
    )

selected_trial_name = str(selected_trial["trial_name"])
selected_model = trained_candidates[selected_trial_name]
selected_probabilities = validation_prediction_columns[selected_trial_name]
selected_threshold = float(selected_trial["threshold"])
selected_predictions = (selected_probabilities >= selected_threshold).astype(np.int8)

print("Tüm tekil ağır model denemeleri:")
display(optimization_results)
print("\nSeçilen model:", selected_trial_name)
print("Seçilen threshold:", selected_threshold)
print("Seçim nedeni:", model_selection_reason)


## 8. Overfit ve eşik kontrolü

Train–validation PR-AUC/ROC-AUC farkı overfit için tanısal olarak raporlanır. Model seçimi test sonucuna dayanmaz. Eşik validation üzerinde sabitlenir ve Görev 7'ye taşınır.


In [ ]:
tn, fp, fn, tp = confusion_matrix(
    validation_target, selected_predictions, labels=[0, 1]
).ravel()
assert (tn, fp, fn, tp) == (
    int(selected_trial["tn"]),
    int(selected_trial["fp"]),
    int(selected_trial["fn"]),
    int(selected_trial["tp"]),
)

overfit_flag = bool(
    float(selected_trial["pr_auc_gap"]) > 0.10
    or float(selected_trial["roc_auc_gap"]) > 0.05
)
selected_model_summary = pd.DataFrame([{
    **selected_trial.to_dict(),
    "model_selection_reason": model_selection_reason,
    "overfit_diagnostic_flag": overfit_flag,
}])

validation_predictions = pd.DataFrame({
    "TransactionID": validation_ids,
    "y_true": validation_target,
    **validation_prediction_columns,
    "selected_trial": selected_trial_name,
    "selected_probability": selected_probabilities,
    "selected_threshold": selected_threshold,
    "selected_prediction": selected_predictions,
})

print(
    f"TP={tp:,} | FP={fp:,} | FN={fn:,} | TN={tn:,}\n"
    f"Precision={precision_score(validation_target, selected_predictions):.4f} | "
    f"Recall={recall_score(validation_target, selected_predictions):.4f} | "
    f"F1={f1_score(validation_target, selected_predictions):.4f}\n"
    f"Train PR-AUC={selected_trial['train_pr_auc']:.4f} | "
    f"Validation PR-AUC={selected_trial['validation_pr_auc']:.4f} | "
    f"Fark={selected_trial['pr_auc_gap']:.4f}"
)
print("Overfit tanı bayrağı:", overfit_flag)
display(selected_model_summary)


## 9. Karşılaştırma görselleri

In [ ]:
plot_data = optimization_results.sort_values("f1")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(plot_data["trial_name"], plot_data["validation_pr_auc"])
axes[0].set_title("Validation PR-AUC")
axes[0].set_xlabel("PR-AUC")
axes[0].grid(axis="x", alpha=0.25)

axes[1].barh(plot_data["trial_name"], plot_data["f1"])
axes[1].set_title(f"Recall ≥ {MINIMUM_RECALL:.2f} Altında En İyi F1")
axes[1].set_xlabel("F1")
axes[1].grid(axis="x", alpha=0.25)

fig.tight_layout()
comparison_figure_path = FIGURES_DIR / "single_heavy_optimization_comparison.png"
fig.savefig(comparison_figure_path, dpi=170, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for trial_name, curve in threshold_curves.groupby("trial_name"):
    curve = curve.sort_values("recall")
    ax.plot(curve["recall"], curve["precision"], label=trial_name)
ax.axvline(MINIMUM_RECALL, color="black", linestyle="--", label="Minimum recall")
ax.set_title("Tekil Ağır Modeller — Precision/Recall Threshold Noktaları")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
threshold_figure_path = FIGURES_DIR / "single_heavy_threshold_tradeoffs.png"
fig.savefig(threshold_figure_path, dpi=170, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Görseller:")
print(" -", comparison_figure_path.relative_to(PROJECT_ROOT))
print(" -", threshold_figure_path.relative_to(PROJECT_ROOT))


## 10. Kazanan modeli ve deney çıktılarını kaydetme

In [ ]:
results_path = PATHS.metrics / "single_heavy_optimization_validation.csv"
curves_path = PATHS.metrics / "single_heavy_threshold_curves.csv"
selected_path = PATHS.metrics / "selected_single_heavy_model.csv"
predictions_path = PREDICTIONS_DIR / "single_heavy_optimization_validation_predictions.parquet"
metadata_path = PATHS.metadata / "single_heavy_optimization_metadata.json"
model_path = MODEL_DIR / "optimized_single_heavy_model.joblib"

optimization_results.to_csv(results_path, index=False)
threshold_curves.to_csv(curves_path, index=False)
selected_model_summary.to_csv(selected_path, index=False)
validation_predictions.to_parquet(predictions_path, index=False, compression="zstd")

print("Kazanan model kaydediliyor:", model_path.name)
save_started = time.perf_counter()
joblib.dump(selected_model, model_path, compress=3)
model_size_mb = model_path.stat().st_size / (1024 ** 2)
print(
    f"Model kaydı tamamlandı: {time.perf_counter() - save_started:.1f} sn | "
    f"{model_size_mb:.2f} MB"
)
if model_size_mb > 100:
    print("UYARI: Model 100 MB'den büyük; Git repository'ye commit etmeyin.")

metadata = {
    "experiment_stage": "task_6_single_heavy_optimization",
    "split_version": "common_v2",
    "training_dataset": "train",
    "selection_dataset": "validation",
    "test_used": False,
    "light_filter_used": False,
    "resampling_used": False,
    "random_state": RANDOM_STATE,
    "minimum_recall": MINIMUM_RECALL,
    "model_selection_rule": model_selection_reason,
    "selected_trial": selected_model_summary.iloc[0].to_dict(),
    "model_path": str(model_path.relative_to(PROJECT_ROOT)),
    "model_size_mb": model_size_mb,
    "trials": optimization_results.to_dict(orient="records"),
}
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
        default=lambda value: value.item()
        if isinstance(value, np.generic)
        else str(value),
    )

print("Kaydedilen dosyalar:")
for path in [
    results_path,
    curves_path,
    selected_path,
    predictions_path,
    metadata_path,
    model_path,
    comparison_figure_path,
    threshold_figure_path,
]:
    print(" -", path.relative_to(PROJECT_ROOT))


## 11. Bellek temizliği

Yalnızca kaydedilmiş kazanan model korunur. Notebook içindeki diğer aday nesneleri ve cache serbest bırakılır.


In [ ]:
for trial_name, candidate in list(trained_candidates.items()):
    if trial_name != selected_trial_name:
        del candidate
trained_candidates.clear()
del catboost_data
gc.collect()
print("Temizlik tamamlandı. RAM:", f"{ram_gb():.2f} GB")


# Görev 6 tamamlanma koşulları

- [x] Hafif model filtresi kullanılmaz.
- [x] Ağır modeller ortak train splitinin tamamıyla eğitilir.
- [x] XGBoost ve CatBoost için kontrollü hiperparametre denemeleri yapılır.
- [x] Preprocessing yalnızca daha önce train üzerinde fit edilmiş ortak cache'lerden gelir.
- [x] Model ve threshold seçimi yalnızca validation üzerinde yapılır.
- [x] Test split yüklenmez ve kullanılmaz.
- [x] Train–validation PR-AUC ve ROC-AUC farkları raporlanır.
- [x] Minimum recall koşulunda precision, F1, FP ve FN kaydedilir.
- [x] Eğitim süresi, inference süresi, RAM ve model boyutu kaydedilir.
- [x] Yalnızca kazanan model yeni dosya adıyla kaydedilir.
- [x] 100 MB üzerindeki model dosyası için commit uyarısı gösterilir.

Görev 6 sonuçları incelendikten sonra Görev 7'de seçilmiş cascade ve tekil ağır model ortak test splitinde yalnızca bir kez karşılaştırılacaktır.
